# 04 — Stacking (통합)

**Base pool**: 가용 base 자동 탐지 (zit, bag_zit, reg_single, two_stage default 등 — POOL_PATHS에 등록된 것 중 산출물 있는 것만 사용. 누락 자동 제외).

**Meta-learner**: `Pipeline(StandardScaler → ElasticNetCV)`
- `alpha`: `np.logspace(-6, 0, 30)` 그리드
- `l1_ratio`: `[0.1, 0.3, 0.5, 0.7, 0.9, 1.0]` 그리드
- 5-fold CV, `positive=False` (음수 weight 허용 — corrector 역할)
- 예측 후 `np.clip(0, None)`

**비교 baseline**: SLSQP blending (`Σw=1, w≥0`)

## 옵션 토글

| 옵션 | 변수 | 기본 | 위치 | 설명 |
|---|---|---|---|---|
| **A — 핵심 피처 추가** | `USE_EXTRA` | `False` | cell-extra | importance 상위 피처를 메타 입력에 합치기. base가 못 잡은 비선형 신호 보강 |
| **D — Zero clip 후처리** | `USE_ZERO_CLIP` | `False` | cell-zeroclip | stacking pred에 임계값 적용 (`pred < th → 0`). train OOF best th → val 적용 → val 개선 시 채택 |

옵션 ON 시 OUT_DIR이 태그로 구분되어 (`base / extra_K20 / zclip / extra_K20_zclip`) 비교 실험 가능.

**출력**: `4_output/04_stacking/{exp_tag}/`

## 1. 환경 + import

In [2]:
import os, sys, json

# Google Drive 파일 ID — Colab에서 코드/데이터 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID    = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개

try:
    import google.colab
    # Colab: 프로젝트 코드가 없으면 Drive에서 받아 /content/project에 설치
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    # 로컬: setup.py가 두 단계 위에 있음
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all

# meta-learner: ElasticNetCV는 L1+L2 혼합으로 불필요한 base를 자동 0-weight 처리
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler  # ElasticNet은 스케일 의존 → 필수
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from scipy.optimize import minimize  # Blending: SLSQP로 가중치 제약(합=1, 비음수) 최적화

# N_JOBS: ElasticNetCV 내부 CV 병렬도 (외부 base 학습은 각 노트북에서 이미 완료)
N_JOBS = 7

print(f'PROJECT_ROOT = {PROJECT_ROOT}')


setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. Base pool 정의 + 자동 탐지

각 디렉토리에 3개 파일 (`oof_unit.csv`, `val_unit.csv`, `test_unit.csv`) 모두 존재해야 pool 포함. 누락 모델은 자동 제외.

In [3]:
POOL_PATHS = {}

# 1) ZIT family (01_zit) — ZITboost + BagZIT 두 변형
POOL_PATHS['zit_only'] = os.path.join(OUTPUT_DIR, '01_zit', 'zit_only')
POOL_PATHS['bag_zit']  = os.path.join(OUTPUT_DIR, '01_zit', 'bag_zit')

# 2) reg_single 5종 (02_reg_single) — LGBM/XGB/CatBoost/ET(베깅)/ElasticNet(선형)
for n in ['lgbm', 'xgb', 'catboost', 'et', 'enet']:
    POOL_PATHS[f'reg__{n}'] = os.path.join(OUTPUT_DIR, '02_reg_single', n)

# 3) two_stage default 16조합 (4 clf × 4 reg) — combine.ipynb 산출 (combined/ 아래 평탄 구조)
_combined_dir = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'combined')
for clf in ['catboost', 'et', 'lgbm', 'xgb']:
    for reg in ['catboost', 'enet', 'et', 'xgb']:
        POOL_PATHS[f'grid__{clf}_x_{reg}'] = os.path.join(_combined_dir, f'{clf}_x_{reg}')

# 4) two_stage reverse (경로 B — 회귀→집계 경로)
POOL_PATHS['ts_reverse'] = os.path.join(OUTPUT_DIR, '03_two_stage', 'reverse')

REQUIRED = ['oof_unit.csv', 'val_unit.csv', 'test_unit.csv']
EXP_PIN  = {}   # 특정 실험 강제 고정 시: {'reg__lgbm': '001', ...}. 비우면 OOF RMSE best 자동 선택

# 각 base 경로에서 OOF RMSE가 가장 좋은 실험 하위폴더를 자동 선택
# combined/ 처럼 base 바로 아래 파일이 있으면 그대로 사용
def _oof_rmse(exp_dir):
    p = os.path.join(exp_dir, 'oof_unit.csv')
    if not os.path.exists(p): return None
    df = pd.read_csv(p)
    if 'pred' not in df.columns or 'health' not in df.columns: return None
    d = df.dropna(subset=['pred', 'health'])
    if len(d) == 0: return None
    return float(np.sqrt(np.mean((d['pred'].values - d['health'].values) ** 2)))

def _resolve_base(base, pin=None):
    if not os.path.isdir(base): return None
    if pin:
        p = os.path.join(base, pin)
        return p if all(os.path.exists(os.path.join(p, f)) for f in REQUIRED) else None
    # base 바로 아래에 REQUIRED 파일이 있으면 구 구조로 바로 사용
    if all(os.path.exists(os.path.join(base, f)) for f in REQUIRED):
        return base
    # hp/ pphp/ raw/ 세 카테고리 모두 후보로 수집 → OOF RMSE 최소 선택
    cands = []
    for cat in ("hp", "pphp", "raw"):
        cat_dir = os.path.join(base, cat)
        if not os.path.isdir(cat_dir): continue
        if all(os.path.exists(os.path.join(cat_dir, f)) for f in REQUIRED):
            cands.append((cat, cat_dir, _oof_rmse(cat_dir)))
            continue
        for d in sorted(os.listdir(cat_dir)):
            sub = os.path.join(cat_dir, d)
            if not (d.isdigit() and os.path.isdir(sub)): continue
            if not all(os.path.exists(os.path.join(sub, f)) for f in REQUIRED): continue
            cands.append((f"{cat}/{d}", sub, _oof_rmse(sub)))
    if not cands: return None
    scored = [c for c in cands if c[2] is not None]
    if scored: return min(scored, key=lambda c: c[2])[1]  # RMSE 최소
    return max(cands, key=lambda c: c[0])[1]  # 점수 없으면 마지막 것

available, missing = {}, {}
for name, base in POOL_PATHS.items():
    resolved = _resolve_base(base, EXP_PIN.get(name))
    if resolved is not None:
        available[name] = resolved
    else:
        missing[name] = base

print('=== Pool 자동 탐지 (실험 선택: OOF RMSE best) ===')
print(f'  사용 가능 ({len(available)}/{len(POOL_PATHS)}):')
for n, p in available.items():
    print(f'    OK  {n:24s} -> {os.path.relpath(p, OUTPUT_DIR)}')
if missing:
    print(f'  누락 ({len(missing)}):')
    for n, base in missing.items():
        print(f'    --  {n}  ({os.path.relpath(base, OUTPUT_DIR)})')

if len(available) < 2:
    raise RuntimeError('stacking 가능한 base < 2개')


=== Pool 자동 탐지 (실험 선택: OOF RMSE best) ===
  사용 가능 (24/24):
    OK  zit_only                 -> 01_zit\zit_only\001
    OK  bag_zit                  -> 01_zit\bag_zit\001
    OK  reg__lgbm                -> 02_reg_single\lgbm\002
    OK  reg__xgb                 -> 02_reg_single\xgb\001
    OK  reg__catboost            -> 02_reg_single\catboost\001
    OK  reg__et                  -> 02_reg_single\et\002
    OK  reg__enet                -> 02_reg_single\enet\001
    OK  grid__catboost_x_catboost -> 03_two_stage\default\combined\catboost_x_catboost
    OK  grid__catboost_x_enet    -> 03_two_stage\default\combined\catboost_x_enet
    OK  grid__catboost_x_et      -> 03_two_stage\default\combined\catboost_x_et
    OK  grid__catboost_x_xgb     -> 03_two_stage\default\combined\catboost_x_xgb
    OK  grid__et_x_catboost      -> 03_two_stage\default\combined\et_x_catboost
    OK  grid__et_x_enet          -> 03_two_stage\default\combined\et_x_enet
    OK  grid__et_x_et            -> 03_two_stage

## 3. OOF/val/test 로드 + 정합 검증

각 csv는 `[ufs_serial, pred, health]` (또는 `[ufs_serial, prob, pred, health]`) 컬럼. `pred` 만 사용. 모든 base 동일 ufs_serial set.

In [4]:
_, ys = load_all()

# 첫 번째 base의 oof_unit.csv에서 health 레이블 가져옴 (CLIP_Y_EXTREME 값 반영됨)
first_name = next(iter(available))
first_oof  = pd.read_csv(os.path.join(available[first_name], 'oof_unit.csv'))
first_val  = pd.read_csv(os.path.join(available[first_name], 'val_unit.csv'))
first_test = pd.read_csv(os.path.join(available[first_name], 'test_unit.csv'))
y_oof  = first_oof.set_index(KEY_COL)['health']
y_val  = first_val.set_index(KEY_COL)['health']
y_test = first_test.set_index(KEY_COL)['health']

print(f'y counts: oof={len(y_oof):,}, val={len(y_val):,}, test={len(y_test):,}')
print(f'y max (clipped): oof={y_oof.max():.6f}, val={y_val.max():.6f}, test={y_test.max():.6f}')

# 모든 base 모델 pred 로드 — {model_name: Series(index=ufs_serial)}
oofs, vals, tests = {}, {}, {}
for n, base in available.items():
    oofs[n]  = pd.read_csv(os.path.join(base, 'oof_unit.csv')).set_index(KEY_COL)['pred']
    vals[n]  = pd.read_csv(os.path.join(base, 'val_unit.csv')).set_index(KEY_COL)['pred']
    tests[n] = pd.read_csv(os.path.join(base, 'test_unit.csv')).set_index(KEY_COL)['pred']

# index 정합 검증 — 불일치 base는 warn 후 drop (stacking 행렬에서 NaN 전파 방지)
def _check_index(d, ref, label):
    drop = []
    for n, s in d.items():
        miss  = ref.difference(s.index)
        extra = s.index.difference(ref)
        if len(miss) or len(extra):
            print(f'  [WARN] {label}/{n}: missing={len(miss)}, extra={len(extra)} → drop')
            drop.append(n)
    return drop

_drop = set()
_drop |= set(_check_index(oofs,  y_oof.index,  'oof'))
_drop |= set(_check_index(vals,  y_val.index,  'val'))
_drop |= set(_check_index(tests, y_test.index, 'test'))

if _drop:
    print(f'\n  index mismatch 로 drop: {sorted(_drop)}')
    for n in _drop:
        oofs.pop(n, None); vals.pop(n, None); tests.pop(n, None)
        available.pop(n, None)

# 정렬된 prediction matrix — 행=unit, 열=base_model
P_oof  = pd.DataFrame({n: oofs[n].reindex(y_oof.index)   for n in available})
P_val  = pd.DataFrame({n: vals[n].reindex(y_val.index)   for n in available})
P_test = pd.DataFrame({n: tests[n].reindex(y_test.index) for n in available})

# NaN 검사: reindex 후 NaN이 있으면 집계 오류 가능성 있음
for label, df in [('oof', P_oof), ('val', P_val), ('test', P_test)]:
    n_nan = df.isna().sum().sum()
    if n_nan > 0:
        print(f'  [WARN] {label}: NaN {n_nan}개 발견')

print(f'\n[정합 OK] P_oof={P_oof.shape}, P_val={P_val.shape}, P_test={P_test.shape}')
print(f'  최종 base 수: {len(available)}')


[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
y counts: oof=26,187, val=8,727, test=8,729
y max (clipped): oof=0.097417, val=0.172211, test=0.602242

[정합 OK] P_oof=(26187, 24), P_val=(8727, 24), P_test=(8729, 24)
  최종 base 수: 24


## 3.5 옵션 A — 핵심 피처 추가 (Extra Features)

`USE_EXTRA=True`이면 importance source 노트북의 fold_models 평균에서 상위 K개 feature를 뽑아 die→unit 집계 (`mean`, `std` 등) 후 P_oof/P_val/P_test에 합친다.

- `IMPORTANCE_SOURCE`: base 디렉토리 (`fold_models.pkl` + `feature_names` 필요). 보통 `reg_single/lgbm` 같이 안정적인 트리 base
- `EXTRA_K`: 상위 몇 개 피처
- `EXTRA_AGG`: die→unit 집계 함수 list (`mean`만, `mean+std`, 등)
- 전처리: 결측은 컬럼 median으로 채움. StandardScaler가 자동으로 base prediction과 스케일 통일

**위험**: 메타가 raw 피처에만 의존하면 base prediction 무시될 수 있음 → ElasticNetCV의 `l1_ratio` 그리드(0.1~1.0)가 sparse 자동 조정.

In [5]:
# 옵션 A — 핵심 피처 추가 (단순 토글)
USE_EXTRA = False              # True로 켜면 raw 피처를 메타 입력에 합침 (단일 source)
EXTRA_K   = 20                 # 상위 K 피처
EXTRA_AGG = ['mean', 'std']    # die→unit 집계 함수 list

# importance source — fold_models.pkl + feature_names 가지고 있는 안정적인 트리 base (1개)
IMPORTANCE_SOURCE = _resolve_base(os.path.join(OUTPUT_DIR, '02_reg_single', 'lgbm'))  # OOF RMSE best 실험 자동 선택 (구 구조면 base 그대로)

# 옵션 A_OPT — Optuna 통합 (importance 가중치 + K + agg + SHAP source 모두 탐색)
USE_OPTUNA_EXTRA  = False                  # True 켜면 cell-shap-cache + cell-extra-optuna 동작
OPTUNA_N_TRIALS   = 1                    # Optuna trial 수 (100=1h, 200=2~3h)
OPTUNA_TIMEOUT    = None                   # 초 단위, None=무제한

# 충돌 체크: 두 모드 동시 ON 금지 (USE_OPTUNA_EXTRA가 우선)
if USE_OPTUNA_EXTRA and USE_EXTRA:
    print(f'[WARN] USE_OPTUNA_EXTRA=True 우선 → USE_EXTRA 자동 OFF')
    USE_EXTRA = False

extra_unit = None
extra_feat_names = []

if USE_EXTRA:
    import pickle
    fm_path = os.path.join(IMPORTANCE_SOURCE, 'fold_models.pkl')
    if not os.path.exists(fm_path):
        raise FileNotFoundError(f'IMPORTANCE_SOURCE에 fold_models.pkl 없음: {fm_path}')
    with open(fm_path, 'rb') as f:
        fm = pickle.load(f)

    # fold_models 평균 importance
    fold_models = fm.get('fold_models') or fm.get('models') or fm
    feat_names  = fm.get('feature_names') or fm.get('feat_cols')
    if feat_names is None:
        raise KeyError('feature_names / feat_cols 누락 (fold_models.pkl 구조 확인)')

    imps = []
    for m in fold_models:
        if hasattr(m, 'feature_importances_'):
            imps.append(m.feature_importances_)
        elif hasattr(m, 'get_feature_importance'):
            imps.append(m.get_feature_importance())
    if not imps:
        raise RuntimeError('fold_models에 feature_importances_ 없음')
    imp_mean = np.mean(imps, axis=0)
    top_idx = np.argsort(-imp_mean)[:EXTRA_K]
    top_feats = [feat_names[i] for i in top_idx]
    print(f'[EXTRA] importance source: {IMPORTANCE_SOURCE}')
    print(f'[EXTRA] top {EXTRA_K} feat (importance 평균):')
    for i, f_ in enumerate(top_feats[:10]):
        print(f'  {i+1:2d}. {f_:10s}  imp={imp_mean[top_idx[i]]:.4f}')
    if EXTRA_K > 10:
        print(f'  ... (+{EXTRA_K - 10}개)')

    # die→unit 집계
    xs_full, _ = load_all()
    extra_die = xs_full[[KEY_COL] + top_feats].copy()
    agg_dict = {f: EXTRA_AGG for f in top_feats}
    extra_unit = extra_die.groupby(KEY_COL).agg(agg_dict)
    extra_unit.columns = [f'X_{c[0]}__{c[1]}' for c in extra_unit.columns]
    # NaN imputation (column median)
    extra_unit = extra_unit.fillna(extra_unit.median())
    extra_feat_names = list(extra_unit.columns)
    print(f'\n[EXTRA] unit 집계 컬럼 {len(extra_feat_names)}개 ({EXTRA_K} feat × {len(EXTRA_AGG)} agg)')

    # P_oof/val/test에 합치기 — KEY_COL index에 정렬
    P_oof  = P_oof.join(extra_unit, how='left')
    P_val  = P_val.join(extra_unit, how='left')
    P_test = P_test.join(extra_unit, how='left')

    # 합친 후 NaN 검사 (어떤 unit이 xs_full에 없을 수 있음)
    for label, df in [('oof', P_oof), ('val', P_val), ('test', P_test)]:
        n_nan = df[extra_feat_names].isna().sum().sum()
        if n_nan > 0:
            print(f'  [WARN] {label}: extra 컬럼에 NaN {n_nan}개 → 0으로 채움')
            df[extra_feat_names] = df[extra_feat_names].fillna(0)

    print(f'\n[EXTRA 합친 후] P_oof={P_oof.shape}, P_val={P_val.shape}, P_test={P_test.shape}')
elif USE_OPTUNA_EXTRA:
    print('[EXTRA] USE_OPTUNA_EXTRA=True → cell-shap-cache + cell-extra-optuna 단계에서 처리')
else:
    print('[EXTRA] 모두 OFF → base prediction만 메타 입력으로 사용')

[EXTRA] 모두 OFF → base prediction만 메타 입력으로 사용


## 3.6 SHAP 캐시 사전 계산 (옵션 A_OPT 전용)

`USE_OPTUNA_EXTRA=True`일 때만 동작. `4_output/04_stacking/_cache/shap_{model}.csv` 3개를 만들어 둔다.

- 캐시 있으면 **즉시 skip** (재실험 비용 0)
- 없으면 1차 백업 또는 신규 reg_single의 fold_models.pkl로 TreeExplainer 계산 → 정규화 후 csv 저장

각 source는 `[feature, shap_score]` 컬럼. shap_score는 fold별 `mean(|shap|)` 평균을 정규화 (`/sum`).

**경로 정책**: 현재 cell-save가 1차 패턴 (`04_stacking/_cache/`)이라 이쪽으로 통일. strategy_common §17 마이그레이션 시 `04_stacking/_cache/`로 일괄 변경.

In [6]:
# SHAP 캐시 사전 계산 (옵션 A_OPT 전용)
SHAP_SOURCES = {
    'lgbm':     os.path.join(OUTPUT_DIR, 'final', 'reg_single', 'lgbm'),
    'xgb':      os.path.join(OUTPUT_DIR, 'final', 'reg_single', 'xgb'),
    'catboost': os.path.join(OUTPUT_DIR, 'final', 'reg_single', 'catboost'),
}
SHAP_CACHE_DIR = os.path.join(OUTPUT_DIR, 'final', 'stacking', '_cache')

# gain importance source (3 base 정규화 평균 — Optuna A_OPT 통합용)
GAIN_SOURCES = list(SHAP_SOURCES.values())

# fallback PP — base의 best_params.json에 effective_pp_params 없을 때 사용 (strategy_common §1)
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# Optuna trial 입력으로 쓸 신호 dict — cell-extra-optuna에서 채움
shap_scores = {}     # {'lgbm': pd.Series(feature -> normalized score), ...}
gain_score  = None   # pd.Series — 3 base 정규화 평균


def _ensure_modeling_imports():
    """preprocess + utils.data lazy import (USE_OPTUNA_EXTRA True일 때만 필요)."""
    pp_dir  = os.path.join(PROJECT_ROOT, '2_preprocessing')
    mod_dir = os.path.join(PROJECT_ROOT, '3_modeling')
    if pp_dir  not in sys.path: sys.path.insert(0, pp_dir)
    if mod_dir not in sys.path: sys.path.insert(0, mod_dir)
    from modules import preprocess as _pp
    from utils.data import get_feat_cols as _gfc, split_xs as _sx
    return _pp, _gfc, _sx


def _load_fold_models(src_dir):
    """fold_models.pkl 로드 → (fold_models, feat_names)."""
    import pickle
    fm_path = os.path.join(src_dir, 'fold_models.pkl')
    if not os.path.exists(fm_path):
        raise FileNotFoundError(f'fold_models.pkl 없음: {fm_path}')
    with open(fm_path, 'rb') as f:
        fm = pickle.load(f)
    fold_models = fm.get('fold_models') or fm.get('models') or fm
    feat_names  = fm.get('feature_names') or fm.get('feat_cols')
    if feat_names is None and hasattr(fold_models[0], 'feature_name_'):
        feat_names = list(fold_models[0].feature_name_)
    if feat_names is None:
        raise KeyError(f'feature_names 누락: {src_dir}')
    return fold_models, feat_names


def _build_xs_pp_for_source(src_dir, _pp_module, _gfc, _sx):
    """source의 best_params.json에서 effective_pp_params 추출 → preprocess.run()
    → xs_train + xs_val + xs_test concat (174K die-level, 모델 학습 동일 feature space)."""
    bp_path = os.path.join(src_dir, 'best_params.json')
    pp_params = PP_FIXED  # fallback
    if os.path.exists(bp_path):
        with open(bp_path, 'r', encoding='utf-8') as f:
            bp = json.load(f)
        pp_params = bp.get('effective_pp_params') or bp.get('effective_params') or PP_FIXED

    xs_full_raw, ys_full = load_all()
    feat_cols = _gfc(xs_full_raw)
    xs_dict   = _sx(xs_full_raw)
    pp = _pp_module.run(xs_full_raw, ys_full, feat_cols, xs_dict, params=pp_params)
    xs_pp_all = pd.concat([pp['xs_train'], pp['xs_val'], pp['xs_test']], axis=0)
    return xs_pp_all


if USE_OPTUNA_EXTRA:
    os.makedirs(SHAP_CACHE_DIR, exist_ok=True)
    print(f'[SHAP] cache dir: {SHAP_CACHE_DIR}')

    _pp_module = _gfc = _sx = None  # lazy import flag

    # 캐시 확인 / 생성
    for name, src in SHAP_SOURCES.items():
        cache_path = os.path.join(SHAP_CACHE_DIR, f'shap_{name}.csv')
        if os.path.exists(cache_path):
            print(f'  [{name}] 캐시 hit → skip')
            continue
        if not os.path.exists(src):
            print(f'  [{name}] source 없음 ({src}) → SHAP skip')
            continue

        print(f'  [{name}] 캐시 miss → preprocess.run() + TreeExplainer 시작...')
        try:
            import shap
        except ImportError:
            raise ImportError('shap 미설치 — `pip install shap` 후 재실행')

        if _pp_module is None:
            _pp_module, _gfc, _sx = _ensure_modeling_imports()

        # source의 effective_pp_params로 전처리 → 모델 학습 동일 feature space
        xs_pp_all = _build_xs_pp_for_source(src, _pp_module, _gfc, _sx)
        fold_models, feat_names = _load_fold_models(src)

        missing_cols = set(feat_names) - set(xs_pp_all.columns)
        if missing_cols:
            print(f'    [WARN] feat_names 중 {len(missing_cols)}개가 전처리 결과에 없음 → 0으로 채움 (예: {list(missing_cols)[:3]})')
        X_die = xs_pp_all.reindex(columns=feat_names, fill_value=0).fillna(0).values
        print(f'    X_die={X_die.shape}, fold_models={len(fold_models)}')

        shap_per_fold = []
        for i, m in enumerate(fold_models):
            try:
                explainer = shap.TreeExplainer(m)
                sv = explainer.shap_values(X_die)
                shap_per_fold.append(np.mean(np.abs(sv), axis=0))
                print(f'    fold {i+1}/{len(fold_models)} 완료')
            except Exception as e:
                print(f'    fold {i+1} 실패: {e}')

        if not shap_per_fold:
            raise RuntimeError(f'[{name}] 모든 fold SHAP 실패')

        shap_avg = np.mean(shap_per_fold, axis=0)
        shap_norm = shap_avg / shap_avg.sum() if shap_avg.sum() > 0 else shap_avg

        df_shap = pd.DataFrame({'feature': feat_names, 'shap_score': shap_norm})
        df_shap.to_csv(cache_path, index=False)
        print(f'  [{name}] 캐시 저장 ({len(feat_names)}개 feat) → {cache_path}')

    # 캐시 로드 (계산 끝났거나 이미 있는 것 모두)
    for name in SHAP_SOURCES:
        cache_path = os.path.join(SHAP_CACHE_DIR, f'shap_{name}.csv')
        if os.path.exists(cache_path):
            df_ = pd.read_csv(cache_path)
            shap_scores[name] = df_.set_index('feature')['shap_score']
    print(f'\n[SHAP] 로드 완료: {list(shap_scores.keys())}')

    # gain importance 3 base 정규화 평균
    print(f'\n[GAIN] 3 base importance 추출...')
    gain_per_src = []
    for src in GAIN_SOURCES:
        if not os.path.exists(src):
            print(f'  source 없음: {src} → gain skip')
            continue
        fold_models, feat_names = _load_fold_models(src)
        imps = []
        for m in fold_models:
            if hasattr(m, 'feature_importances_'):
                imps.append(m.feature_importances_)
            elif hasattr(m, 'get_feature_importance'):
                imps.append(m.get_feature_importance())
        if not imps:
            continue
        imp = np.mean(imps, axis=0)
        imp_norm = imp / imp.sum() if imp.sum() > 0 else imp
        s = pd.Series(imp_norm, index=feat_names)
        gain_per_src.append(s)
        print(f'  [{os.path.basename(src)}] feat={len(feat_names)}, top1={s.idxmax()} (norm={s.max():.4f})')

    if gain_per_src:
        # 모든 src의 feature union으로 정렬 → 평균
        gain_score = pd.concat(gain_per_src, axis=1).fillna(0).mean(axis=1)
        gain_score = gain_score / gain_score.sum() if gain_score.sum() > 0 else gain_score
        print(f'[GAIN] 통합 완료: {len(gain_score)}개 feat')
    else:
        raise RuntimeError('[GAIN] 사용 가능한 base 0개')
else:
    print('[SHAP/GAIN] USE_OPTUNA_EXTRA=False → skip')

[SHAP/GAIN] USE_OPTUNA_EXTRA=False → skip


## 3.7 옵션 A_OPT — Optuna 통합 importance + K + agg + SHAP source

`USE_OPTUNA_EXTRA=True`일 때만 동작. cell-stack은 자동 패스되며, 여기서 **best trial 재학습까지 끝내 stack_oof/val/test 직접 채운다** (cell-compare/save는 그대로 동작).

**탐색 축**:
- `w_shap` (0~1) + `shap_lgbm/xgb/cb` (0~1) — gain/SHAP 가중치 + 3 SHAP source 혼합
- `K` (10~100 step 10) — top-K 피처
- `agg_{name}` 7개 binary — `mean / std / max / min / range / Q25 / Q75`

**1 trial = ElasticNetCV 1회** (`n_jobs=1`로 옆 세션 영향 최소). 100 trial ≈ 50~100분.

**산출물 변수**:
- `OPTUNA_BEST_EXTRA` dict — best params/aggs/top_feats
- `extra_imp_df` — gain/shap/final + in_top 분해 (cell-save에서 csv 저장)
- `stack_oof/val/test`, `rmse_stack_*`, `stack_coef`, `n_active`, `n_neg`, `best_alpha/l1_ratio` — cell-stack 변수 호환

In [7]:
# 옵션 A_OPT — Optuna 통합 (USE_OPTUNA_EXTRA 전용)
OPTUNA_BEST_EXTRA = None
extra_imp_df = None
AGG_NAMES = ['mean', 'std', 'max', 'min', 'range', 'Q25', 'Q75']

if USE_OPTUNA_EXTRA:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner

    if 'xs_full' not in dir():
        xs_full, _ = load_all()

    # base prediction만 보존 (extra 합치기 전)
    P_oof_base  = P_oof.copy()
    P_val_base  = P_val.copy()
    P_test_base = P_test.copy()

    def _build_extra_unit(top_feats, aggs):
        """top_feats × aggs로 die→unit 집계 → DataFrame (KEY_COL index)."""
        normal_aggs = [a for a in aggs if a not in ('range',)]
        agg_dict = {}
        for f_ in top_feats:
            funcs = []
            for a in normal_aggs:
                if a == 'Q25':
                    funcs.append(('Q25', lambda x: x.quantile(0.25)))
                elif a == 'Q75':
                    funcs.append(('Q75', lambda x: x.quantile(0.75)))
                else:
                    funcs.append(a)
            agg_dict[f_] = funcs
        extra_die = xs_full[[KEY_COL] + list(top_feats)].copy()
        u = extra_die.groupby(KEY_COL).agg(agg_dict)
        u.columns = [f'X_{c[0]}__{c[1]}' for c in u.columns]
        if 'range' in aggs:
            u_max = extra_die.groupby(KEY_COL).max()
            u_min = extra_die.groupby(KEY_COL).min()
            u_rng = (u_max - u_min)
            u_rng.columns = [f'X_{c}__range' for c in u_rng.columns]
            u = pd.concat([u, u_rng], axis=1)
        return u.fillna(u.median())

    def _compose_importance(w_shap, shap_mix):
        """gain + (3 source SHAP 혼합) → final_imp pd.Series."""
        sum_s = sum(shap_mix.values())
        if sum_s < 1e-6:
            return None
        shap_combined = sum(shap_mix[n] * shap_scores[n] / sum_s for n in shap_mix)
        all_feat = gain_score.index.union(shap_combined.index)
        gain_a = gain_score.reindex(all_feat).fillna(0)
        shap_a = shap_combined.reindex(all_feat).fillna(0)
        return (1 - w_shap) * gain_a + w_shap * shap_a

    y_oof_arr_  = y_oof.values
    y_val_arr_  = y_val.values
    y_test_arr_ = y_test.values

    def objective(trial):
        # importance 가중치
        w_shap = trial.suggest_float('w_shap', 0.0, 1.0)
        shap_mix = {n: trial.suggest_float(f'shap_{n}', 0.0, 1.0) for n in shap_scores}

        final_imp = _compose_importance(w_shap, shap_mix)
        if final_imp is None:
            raise optuna.TrialPruned()

        # K, agg
        K = trial.suggest_int('K', 10, 100, step=10)
        aggs = [a for a in AGG_NAMES if trial.suggest_categorical(f'agg_{a}', [True, False])]
        if not aggs:
            raise optuna.TrialPruned()

        top_feats = final_imp.nlargest(K).index.tolist()

        try:
            u = _build_extra_unit(top_feats, aggs)
        except Exception:
            raise optuna.TrialPruned()

        Po = P_oof_base.join(u, how='left').fillna(0)
        Pv = P_val_base.join(u, how='left').fillna(0)
        Pt = P_test_base.join(u, how='left').fillna(0)

        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('enet', ElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                alphas=np.logspace(-6, 0, 30),
                cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
                random_state=SEED, n_jobs=1, max_iter=20000, positive=False,
            )),
        ])
        try:
            pipe.fit(Po.values, y_oof_arr_)
        except Exception:
            raise optuna.TrialPruned()

        en = pipe.named_steps['enet']
        # objective는 ElasticNetCV의 내부 5-fold CV best mse 기준 (정직한 meta-CV 추정).
        # val/test는 진단용 user_attr만 — strategy_common §19: train OOF best → val 적용 → 개선 시 채택.
        # mse_path_ shape: (n_l1_ratio, n_alphas, n_folds). best_combo의 fold 평균 = 선택된 cv mse.
        cv_mse = float(en.mse_path_.mean(axis=-1).min())
        cv_rmse = float(np.sqrt(cv_mse))

        # 진단용 (objective 영향 없음)
        val_pred  = np.clip(pipe.predict(Pv.values), 0, None)
        oof_pred  = np.clip(pipe.predict(Po.values), 0, None)   # in-sample (over-optimistic, 진단만)
        test_pred = np.clip(pipe.predict(Pt.values), 0, None)
        val_rmse  = float(np.sqrt(np.mean((val_pred  - y_val_arr_)**2)))
        oof_rmse_insample = float(np.sqrt(np.mean((oof_pred  - y_oof_arr_)**2)))
        test_rmse = float(np.sqrt(np.mean((test_pred - y_test_arr_)**2)))

        trial.set_user_attr('cv_rmse',           cv_rmse)         # objective 값과 동일
        trial.set_user_attr('val_rmse',          val_rmse)        # 진단 (cherry-pick 금지)
        trial.set_user_attr('test_rmse',         test_rmse)       # 진단
        trial.set_user_attr('oof_rmse_insample', oof_rmse_insample)  # 진단 (in-sample)
        trial.set_user_attr('K_actual',          int(K))
        trial.set_user_attr('aggs',              ','.join(aggs))
        trial.set_user_attr('n_extra_cols',      int(u.shape[1]))
        trial.set_user_attr('alpha',             float(en.alpha_))
        trial.set_user_attr('l1_ratio',          float(en.l1_ratio_))
        return cv_rmse

    optuna_db = os.path.join(SHAP_CACHE_DIR, 'optuna_extra.db')
    sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=10)
    pruner  = MedianPruner(n_warmup_steps=10)
    study = optuna.create_study(
        direction='minimize',
        study_name='stacking_extra_optuna',
        storage=f'sqlite:///{optuna_db}',
        load_if_exists=True,
        sampler=sampler,
        pruner=pruner,
    )
    print(f'\n[OPTUNA] start (n_trials={OPTUNA_N_TRIALS}, timeout={OPTUNA_TIMEOUT}, db={optuna_db})')
    study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT,
                   show_progress_bar=True)

    best_trial  = study.best_trial
    best_params = dict(best_trial.params)
    print(f'\n[OPTUNA best] val RMSE = {best_trial.value:.6f}')
    print(f'  params:  {best_params}')
    print(f'  attrs:   {dict(best_trial.user_attrs)}')

    # best 재학습으로 stack_oof/val/test 채움 (cell-stack 호환 변수)
    w_shap_best  = best_params['w_shap']
    shap_mix_best = {n: best_params[f'shap_{n}'] for n in shap_scores}
    final_imp     = _compose_importance(w_shap_best, shap_mix_best)
    K_best        = best_params['K']
    aggs_best     = [a for a in AGG_NAMES if best_params[f'agg_{a}']]
    top_feats_best = final_imp.nlargest(K_best).index.tolist()

    extra_unit       = _build_extra_unit(top_feats_best, aggs_best)
    extra_feat_names = list(extra_unit.columns)

    # P_oof/val/test에 합치기 (extra OFF 모드와 동일 패턴)
    P_oof  = P_oof.join(extra_unit,  how='left').fillna(0)
    P_val  = P_val.join(extra_unit,  how='left').fillna(0)
    P_test = P_test.join(extra_unit, how='left').fillna(0)

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('enet', ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
            alphas=np.logspace(-6, 0, 30),
            cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
            random_state=SEED, n_jobs=N_JOBS, max_iter=20000, positive=False,
        )),
    ])
    pipe.fit(P_oof.values, y_oof.values)
    en = pipe.named_steps['enet']
    best_alpha     = float(en.alpha_)
    best_l1_ratio  = float(en.l1_ratio_)
    stack_coef     = en.coef_
    stack_intercept = float(en.intercept_)
    n_active = sum(1 for c in stack_coef if abs(c) > 1e-9)
    n_neg    = sum(1 for c in stack_coef if c < -1e-9)

    stack_oof  = np.clip(pipe.predict(P_oof.values),  0, None)
    stack_val  = np.clip(pipe.predict(P_val.values),  0, None)
    stack_test = np.clip(pipe.predict(P_test.values), 0, None)

    rmse_stack_oof  = float(np.sqrt(np.mean((stack_oof  - y_oof.values )**2)))
    rmse_stack_val  = float(np.sqrt(np.mean((stack_val  - y_val.values )**2)))
    rmse_stack_test = float(np.sqrt(np.mean((stack_test - y_test.values)**2)))

    neg_pre_clip_oof  = (pipe.predict(P_oof.values)  < 0).mean()
    neg_pre_clip_val  = (pipe.predict(P_val.values)  < 0).mean()
    neg_pre_clip_test = (pipe.predict(P_test.values) < 0).mean()

    print(f'\n[BEST refit] alpha={best_alpha:.4e}, l1_ratio={best_l1_ratio:.4f}, '
          f'n_active={n_active}/{len(stack_coef)} (음수 {n_neg})')
    print(f'  Stack RMSE: oof={rmse_stack_oof:.6f}, val={rmse_stack_val:.6f}, test={rmse_stack_test:.6f}')

    # importance 분해 csv (cell-save에서 저장)
    all_feat_idx = final_imp.index
    shap_mix_sum = sum(shap_mix_best.values())
    shap_combined_best = sum(shap_mix_best[n] * shap_scores[n] / shap_mix_sum for n in shap_mix_best)
    extra_imp_df = pd.DataFrame({
        'feature': all_feat_idx,
        'gain':    gain_score.reindex(all_feat_idx).fillna(0).values,
        'shap':    shap_combined_best.reindex(all_feat_idx).fillna(0).values,
        'final':   final_imp.values,
        'in_top':  pd.Series(all_feat_idx).isin(top_feats_best).values,
    }).sort_values('final', ascending=False).reset_index(drop=True)

    OPTUNA_BEST_EXTRA = {
        'w_shap':       w_shap_best,
        'shap_mix':     shap_mix_best,
        'K':            int(K_best),
        'aggs':         aggs_best,
        'top_feats':    top_feats_best,
        'best_value':   float(best_trial.value),
        'n_trials':     len(study.trials),
        'optuna_db':    optuna_db,
    }
else:
    print('[A_OPT] USE_OPTUNA_EXTRA=False → skip')

[A_OPT] USE_OPTUNA_EXTRA=False → skip


## 4. 단일 base RMSE + residual correlation

27개라 corr matrix 가 크므로 **NN 과 다른 base 의 residual corr** 만 별도 정렬해서 출력.

In [8]:
def _rmse(p, y):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

rows = []
for n in available:
    rows.append({
        'model': n,
        'oof':   _rmse(P_oof[n].values,  y_oof.values),
        'val':   _rmse(P_val[n].values,  y_val.values),
        'test':  _rmse(P_test[n].values, y_test.values),
    })
single_df = pd.DataFrame(rows).sort_values('val').reset_index(drop=True)
print(f'=== 단일 base RMSE (val 오름차순, top {len(single_df)}) ===')
print(single_df.to_string(index=False, float_format='%.6f'))

# residual corr — base만 (extra 컬럼은 raw 피처값이라 residual 의미 없음)
_base_for_corr = list(available)
R_oof  = P_oof[_base_for_corr].subtract(y_oof,  axis=0)
R_test = P_test[_base_for_corr].subtract(y_test, axis=0)
C_oof  = R_oof.corr()
C_test = R_test.corr()

# NN 과의 잔차 상관 (오름차순)
if 'nn_ft' in available:
    nn_corr_oof  = C_oof['nn_ft'].drop('nn_ft').sort_values()
    nn_corr_test = C_test['nn_ft'].drop('nn_ft').sort_values()
    print(f'\n=== NN vs 다른 base — residual corr (OOF, low → high) ===')
    print(nn_corr_oof.round(4).to_string())

# 가장 보완적인 페어 (OOF)
names_list = list(available)
min_pair = (None, None, 1.0)
for i, a in enumerate(names_list):
    for b in names_list[i+1:]:
        c = C_oof.loc[a, b]
        if c < min_pair[2]:
            min_pair = (a, b, c)
print(f'\n가장 낮은 OOF residual corr: {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')

=== 단일 base RMSE (val 오름차순, top 24) ===
                    model      oof      val     test
                 zit_only 0.005497 0.005704 0.008411
                  bag_zit 0.005499 0.005705 0.008409
               ts_reverse 0.005497 0.005706 0.008412
          grid__lgbm_x_et 0.005499 0.005709 0.008413
           grid__xgb_x_et 0.005499 0.005710 0.008414
      grid__catboost_x_et 0.005503 0.005714 0.008416
            grid__et_x_et 0.005506 0.005721 0.008422
                reg__lgbm 0.005509 0.005722 0.008428
                 reg__xgb 0.005526 0.005728 0.008423
            reg__catboost 0.005522 0.005730 0.008429
                  reg__et 0.005520 0.005734 0.008433
    grid__lgbm_x_catboost 0.005556 0.005756 0.008456
        grid__lgbm_x_enet 0.005544 0.005756 0.008449
     grid__xgb_x_catboost 0.005556 0.005757 0.008457
         grid__xgb_x_enet 0.005545 0.005758 0.008450
grid__catboost_x_catboost 0.005559 0.005761 0.008460
    grid__catboost_x_enet 0.005547 0.005761 0.008452
      

## 5. Blending baseline (SLSQP — Σw=1, w≥0)

Stacking 효과 비교용. 27 base 동일 가중치 시작 → SLSQP 최적화.

In [9]:
K = len(P_oof.columns)            # 메타 입력 차원 (base + extra 합산)
P_oof_arr  = P_oof.values
P_val_arr  = P_val.values
P_test_arr = P_test.values
y_oof_arr  = y_oof.values
y_val_arr  = y_val.values
y_test_arr = y_test.values

# Blending은 base 예측만 의미 있음 — extra 컬럼은 가중평균 해석 불가이므로 base만 사용
_base_names   = list(available)
_P_oof_blend  = P_oof[_base_names].values
_P_val_blend  = P_val[_base_names].values
_P_test_blend = P_test[_base_names].values
K_blend = len(_base_names)

# SLSQP: w·sum=1, w>=0 제약 하에 OOF RMSE 최소화 → 음수 가중치 허용 안 함
res = minimize(
    lambda w: _rmse(_P_oof_blend @ w, y_oof_arr),
    np.full(K_blend, 1.0/K_blend),
    method='SLSQP',
    bounds=[(0.0, 1.0)] * K_blend,
    constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
    options={'ftol': 1e-9, 'maxiter': 1000},
)
w_blend = res.x

blend_oof  = _P_oof_blend  @ w_blend
blend_val  = _P_val_blend  @ w_blend
blend_test = _P_test_blend @ w_blend

rmse_blend_oof  = _rmse(blend_oof,  y_oof_arr)
rmse_blend_val  = _rmse(blend_val,  y_val_arr)
rmse_blend_test = _rmse(blend_test, y_test_arr)

print(f'=== Blending SLSQP ({K_blend} base, converged={res.success}) — top 10 weight ===')
for n, w in sorted(zip(_base_names, w_blend), key=lambda x: -x[1])[:10]:
    bar = '█' * int(w * 60)
    print(f'  {n:32s}: {w:.4f}  {bar}')
n_zero = sum(1 for w in w_blend if w < 1e-6)
print(f'  ... 0-weight base: {n_zero}/{K_blend}')
print(f'\n  OOF RMSE:  {rmse_blend_oof:.6f}')
print(f'  val RMSE:  {rmse_blend_val:.6f}')
print(f'  test RMSE: {rmse_blend_test:.6f}')
print(f'\n  (메타 입력 전체 차원 K={K}: base {K_blend} + extra {K - K_blend})')


=== Blending SLSQP (24 base, converged=True) — top 10 weight ===
  ts_reverse                      : 0.1594  █████████
  zit_only                        : 0.1546  █████████
  bag_zit                         : 0.1441  ████████
  grid__lgbm_x_et                 : 0.1179  ███████
  grid__xgb_x_et                  : 0.1127  ██████
  grid__catboost_x_et             : 0.1085  ██████
  grid__et_x_et                   : 0.0935  █████
  reg__lgbm                       : 0.0586  ███
  reg__et                         : 0.0352  ██
  reg__catboost                   : 0.0154  
  ... 0-weight base: 14/24

  OOF RMSE:  0.005496
  val RMSE:  0.005707
  test RMSE: 0.008412

  (메타 입력 전체 차원 K=24: base 24 + extra 0)


## 6. Stacking — ElasticNetCV meta + StandardScaler

**ElasticNetCV** 5-fold CV로 (alpha, l1_ratio) 자동 선택. l1_ratio 0.9~1.0 이면 Lasso 성격이 강해 redundant base 제거. 음수 weight 허용 (corrector).

In [10]:
if USE_OPTUNA_EXTRA:
    # Optuna extra가 활성화된 경우 이미 meta-learner 학습이 완료됨 → skip
    print('[STACK] USE_OPTUNA_EXTRA=True → cell-extra-optuna에서 이미 처리됨, skip')
    print(f'  Stack RMSE: oof={rmse_stack_oof:.6f}, val={rmse_stack_val:.6f}, test={rmse_stack_test:.6f}')
else:
    # meta-learner: StandardScaler + ElasticNetCV (5-fold, CV로 alpha/l1_ratio 자동 선택)
    pipe = Pipeline([
        ('scaler', StandardScaler()),     # base 예측값 스케일이 제각각 → 정규화 필수
        ('enet',   ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # L1 비중 후보 (0=Ridge, 1=Lasso)
            alphas=np.logspace(-6, 0, 30),              # 정규화 강도 후보 30개
            cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
            random_state=SEED,
            n_jobs=N_JOBS,
            max_iter=20000,
            positive=False,  # 음수 계수 허용 → corrector(base 오차 상쇄) 역할 가능
        )),
    ])
    pipe.fit(P_oof_arr, y_oof_arr)

    en = pipe.named_steps['enet']
    best_alpha    = float(en.alpha_)
    best_l1_ratio = float(en.l1_ratio_)
    stack_coef    = en.coef_
    stack_intercept = float(en.intercept_)

    print(f'=== ElasticNetCV best ===')
    print(f'  alpha    : {best_alpha:.6e}')
    print(f'  l1_ratio : {best_l1_ratio:.4f}')
    print(f'  intercept: {stack_intercept:+.6f}')
    print(f'\n  coef (scaled space, abs 정렬, 비영 항만):')
    coef_names = list(P_oof.columns)
    for n, c in sorted(zip(coef_names, stack_coef), key=lambda x: -abs(x[1])):
        if abs(c) < 1e-9:
            continue
        bar = '█' * int(min(abs(c) * 1500, 50))
        sign = '+' if c >= 0 else '-'
        print(f'    {n:32s}: {sign}{abs(c):.5f}  {bar}')
    n_active = sum(1 for c in stack_coef if abs(c) > 1e-9)
    n_neg    = sum(1 for c in stack_coef if c < -1e-9)
    print(f'  활성 차원: {n_active}/{K} (음수 weight {n_neg}개 = corrector)')

    # clip(0): ElasticNet은 음수 예측 가능 → health는 0 이상이므로 후처리 clip
    stack_oof  = np.clip(pipe.predict(P_oof_arr),  0, None)
    stack_val  = np.clip(pipe.predict(P_val_arr),  0, None)
    stack_test = np.clip(pipe.predict(P_test_arr), 0, None)

    rmse_stack_oof  = _rmse(stack_oof,  y_oof_arr)
    rmse_stack_val  = _rmse(stack_val,  y_val_arr)
    rmse_stack_test = _rmse(stack_test, y_test_arr)

    # clip 전 음수 비율 — 높으면 meta-learner가 오버슈팅 중임을 의미
    neg_pre_clip_oof  = (pipe.predict(P_oof_arr)  < 0).mean()
    neg_pre_clip_val  = (pipe.predict(P_val_arr)  < 0).mean()
    neg_pre_clip_test = (pipe.predict(P_test_arr) < 0).mean()

    print(f'\n=== Stacking RMSE ===')
    print(f'  OOF RMSE:  {rmse_stack_oof:.6f}')
    print(f'  val RMSE:  {rmse_stack_val:.6f}')
    print(f'  test RMSE: {rmse_stack_test:.6f}')
    print(f'  음수 clip 비율: oof={neg_pre_clip_oof:.1%}, val={neg_pre_clip_val:.1%}, test={neg_pre_clip_test:.1%}')

    # NN base가 있으면 계수 별도 출력 (NN 기여도 진단용)
    if 'nn_ft' in available:
        nn_idx = list(P_oof.columns).index('nn_ft')
        nn_coef = stack_coef[nn_idx]
        print(f'\n  [NN] coef = {nn_coef:+.6f}  ({"활성" if abs(nn_coef) > 1e-9 else "0 — 풀에 흡수"})')


=== ElasticNetCV best ===
  alpha    : 1.743329e-05
  l1_ratio : 0.7000
  intercept: +0.002481

  coef (scaled space, abs 정렬, 비영 항만):
    zit_only                        : +0.00051  
    ts_reverse                      : +0.00026  
    grid__xgb_x_et                  : +0.00020  
    reg__xgb                        : -0.00016  
    bag_zit                         : +0.00012  
    grid__catboost_x_et             : +0.00011  
    grid__lgbm_x_et                 : +0.00002  
    reg__enet                       : -0.00001  
    reg__lgbm                       : +0.00001  
    reg__catboost                   : -0.00000  
  활성 차원: 10/24 (음수 weight 3개 = corrector)

=== Stacking RMSE ===
  OOF RMSE:  0.005493
  val RMSE:  0.005704
  test RMSE: 0.008410
  음수 clip 비율: oof=0.0%, val=0.0%, test=0.0%


## 6.5 옵션 D — Zero clip 후처리

`USE_ZERO_CLIP=True`이면 stacking pred에 임계값 적용 (`pred < th → 0`).

- `ZERO_CLIP_GRID`: 후보 th 리스트 (기본 0.001 ~ 0.015 step 0.001)
- train OOF에서 best th 탐색 → val 적용 → **val 개선시에만 채택** (개선 안 되면 미적용)
- 채택되면 `stack_oof / stack_val / stack_test` 덮어씀 (이후 비교/저장 그대로 동작)

**원칙**: zero-inflation 70.8% 환경에서 작은 pred는 false positive 가능성. train OOF best로 선택 → val로 검증 → 안 좋으면 적용 안 함.

In [11]:
# 옵션 D — Zero clip 후처리
USE_ZERO_CLIP   = False                                 # True로 켜면 zero clip 탐색 + val 검증
ZERO_CLIP_GRID  = np.arange(0.001, 0.016, 0.001)        # 후보 th 1e-3 ~ 1.5e-2 step 1e-3

zero_clip_applied  = False
zero_clip_best_th  = None
rmse_stack_oof_pre  = rmse_stack_oof
rmse_stack_val_pre  = rmse_stack_val
rmse_stack_test_pre = rmse_stack_test

if USE_ZERO_CLIP:
    # train OOF best th 탐색
    best_th, best_oof_rmse = None, rmse_stack_oof
    for th in ZERO_CLIP_GRID:
        cand = np.where(stack_oof < th, 0.0, stack_oof)
        r = _rmse(cand, y_oof_arr)
        if r < best_oof_rmse:
            best_th, best_oof_rmse = float(th), r

    if best_th is None:
        print(f'[ZERO_CLIP] OOF에서 개선 임계값 없음 → 미적용')
    else:
        # val 검증
        val_clipped = np.where(stack_val < best_th, 0.0, stack_val)
        r_val_clip = _rmse(val_clipped, y_val_arr)
        print(f'[ZERO_CLIP] OOF best th = {best_th:.4f}')
        print(f'  OOF RMSE: {rmse_stack_oof:.6f} → {best_oof_rmse:.6f}  (Δ={best_oof_rmse-rmse_stack_oof:+.6f})')
        print(f'  val RMSE: {rmse_stack_val:.6f} → {r_val_clip:.6f}     (Δ={r_val_clip-rmse_stack_val:+.6f})')

        if r_val_clip < rmse_stack_val - 1e-7:
            # 채택 — stack_* 덮어쓰기
            stack_oof  = np.where(stack_oof  < best_th, 0.0, stack_oof)
            stack_val  = val_clipped
            stack_test = np.where(stack_test < best_th, 0.0, stack_test)
            rmse_stack_oof  = _rmse(stack_oof,  y_oof_arr)
            rmse_stack_val  = _rmse(stack_val,  y_val_arr)
            rmse_stack_test = _rmse(stack_test, y_test_arr)
            zero_clip_applied = True
            zero_clip_best_th = best_th
            print(f'  → ✓ 채택 (val 개선) → stack_oof/val/test 덮어씀')
            print(f'  최종 OOF RMSE:  {rmse_stack_oof:.6f}')
            print(f'  최종 val RMSE:  {rmse_stack_val:.6f}')
            print(f'  최종 test RMSE: {rmse_stack_test:.6f}')
        else:
            print(f'  → ✗ 미채택 (val 개선 없음 또는 악화) → 원본 stack_* 유지')
else:
    print('[ZERO_CLIP] USE_ZERO_CLIP=False → 미적용')

[ZERO_CLIP] USE_ZERO_CLIP=False → 미적용


## 6.6 Segment 분해 진단 (Y=0 / Y>0)

Zero-inflated target에서 단일 RMSE는 noise를 가림. **Y=0 segment vs Y>0 segment** 분해로 진짜 효과 확인.

전체 val=plateau여도 **Y>0에서 0.001 개선**이면 진짜 효과. 반대로 Y=0만 개선되고 Y>0 악화면 trade-off.

사용자 메모리 정책 (`평가 기준 정책`): "y의 max 분포가 split별로 달라(train 1.0/val 0.17/test 0.60) RMSE 단독 판단 금지, segment 분해 필수".

`segment_rmse` dict는 cell-save의 meta.json `stacking.segment_rmse`에 기록됨.

In [12]:
# Segment 분해 진단
def _seg_rmse(p, y, mask):
    if mask.sum() == 0:
        return float('nan')
    return float(np.sqrt(np.mean((p[mask] - y[mask])**2)))

_y_oof_arr  = y_oof.values
_y_val_arr  = y_val.values
_y_test_arr = y_test.values

segment_rmse = {}
for sp_name, p_arr, y_arr in [
    ('oof',  stack_oof,  _y_oof_arr),
    ('val',  stack_val,  _y_val_arr),
    ('test', stack_test, _y_test_arr),
]:
    m_y0   = (y_arr == 0)
    m_ypos = (y_arr > 0)
    segment_rmse[sp_name] = {
        'y0':     _seg_rmse(p_arr, y_arr, m_y0),
        'ypos':   _seg_rmse(p_arr, y_arr, m_ypos),
        'n_y0':   int(m_y0.sum()),
        'n_ypos': int(m_ypos.sum()),
    }

print('=== Segment 분해 (Stacking pred — Y=0 vs Y>0) ===')
print(f'  {"split":5s}  {"Y=0 RMSE":>11s} (n)         {"Y>0 RMSE":>11s} (n)         전체 RMSE')
for sp_name in ['oof', 'val', 'test']:
    s = segment_rmse[sp_name]
    total = {'oof': rmse_stack_oof, 'val': rmse_stack_val, 'test': rmse_stack_test}[sp_name]
    print(f'  {sp_name:5s}  {s["y0"]:11.6f} (n={s["n_y0"]:>5,})  '
          f'{s["ypos"]:11.6f} (n={s["n_ypos"]:>5,})  {total:11.6f}')

print('\n  → Y>0 segment 개선이 진짜 효과. Y=0만 개선이면 trade-off 가능성.')

=== Segment 분해 (Stacking pred — Y=0 vs Y>0) ===
  split     Y=0 RMSE (n)            Y>0 RMSE (n)         전체 RMSE
  oof       0.002557 (n=18,541)     0.009353 (n=7,646)     0.005493
  val       0.002558 (n=6,178)     0.009773 (n=2,549)     0.005704
  test      0.002544 (n=6,185)     0.015064 (n=2,544)     0.008410

  → Y>0 segment 개선이 진짜 효과. Y=0만 개선이면 trade-off 가능성.


## 7. 종합 비교 표

In [13]:
best_oof_row  = single_df.sort_values('oof').iloc[0]
best_val_row  = single_df.sort_values('val').iloc[0]
best_test_row = single_df.sort_values('test').iloc[0]

# 기존 stacking_11base val (참고용)
STACKING_11BASE_VAL = 0.005701
STACKING_11BASE_TEST = 0.008408

comparison = pd.DataFrame([
    {'method': f'best single (OOF) [{best_oof_row["model"]}]',
     'oof': best_oof_row['oof'], 'val': best_oof_row['val'], 'test': best_oof_row['test']},
    {'method': f'best single (val) [{best_val_row["model"]}]',
     'oof': best_val_row['oof'], 'val': best_val_row['val'], 'test': best_val_row['test']},
    {'method': f'stacking_11base (기존)',
     'oof': float('nan'), 'val': STACKING_11BASE_VAL, 'test': STACKING_11BASE_TEST},
    {'method': f'Blending SLSQP ({K} base)',
     'oof': rmse_blend_oof, 'val': rmse_blend_val, 'test': rmse_blend_test},
    {'method': f'Stacking ElasticNet ({K} base)',
     'oof': rmse_stack_oof, 'val': rmse_stack_val, 'test': rmse_stack_test},
])
print('=' * 100)
print(f'  Stacking Full Pool — 종합 비교 (사용 가능 base = {K}종)')
print('=' * 100)
print(comparison.to_string(index=False, float_format='%.6f'))
print('=' * 100)
# strategy_common §14: stacking_11base 대비 Δ는 진단 스크립트로 분리 (meta.json delta_vs_11base 필드 참조)

  Stacking Full Pool — 종합 비교 (사용 가능 base = 24종)
                       method      oof      val     test
 best single (OOF) [zit_only] 0.005497 0.005704 0.008411
 best single (val) [zit_only] 0.005497 0.005704 0.008411
         stacking_11base (기존)      NaN 0.005701 0.008408
     Blending SLSQP (24 base) 0.005496 0.005707 0.008412
Stacking ElasticNet (24 base) 0.005493 0.005704 0.008410


## 8. 산출물 저장 (`4_output/04_stacking/{exp_tag}/`)

`exp_tag`는 옵션 토글 조합으로 자동 생성:
- `base` — 옵션 모두 OFF
- `extra_K20_mean_std` — A안만 ON (K=20, mean+std)
- `zclip` — D안만 ON
- `extra_K20_mean_std_zclip` — A+D 같이 ON

옵션별로 디렉토리가 분리되어 직접 비교 가능. `meta.json`에 `options` 섹션으로 기록.

In [14]:
# OUT_DIR: 옵션 태그로 디렉토리 구분 (실험 비교 가능)
exp_tag_parts = []
if USE_OPTUNA_EXTRA:
    exp_tag_parts.append('optuna_extra')
elif USE_EXTRA:
    exp_tag_parts.append(f'extra_K{EXTRA_K}_' + '_'.join(EXTRA_AGG))
if USE_ZERO_CLIP:
    exp_tag_parts.append('zclip')
EXP_TAG = '_'.join(exp_tag_parts) if exp_tag_parts else 'base'

OUT_DIR = os.path.join(OUTPUT_DIR, '04_stacking', EXP_TAG)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'OUT_DIR = {OUT_DIR}\n')

def _save_unit(pred_arr, ids, y_arr, fname):
    out = pd.DataFrame({
        KEY_COL: ids,
        'pred':  pred_arr,
        'health': y_arr.reindex(ids).values,
    })
    out.to_csv(os.path.join(OUT_DIR, fname), index=False)

# stacking 산출물
_save_unit(stack_oof,  y_oof.index,  y_oof,  'oof_unit_stack.csv')
_save_unit(stack_val,  y_val.index,  y_val,  'val_unit_stack.csv')
_save_unit(stack_test, y_test.index, y_test, 'test_unit_stack.csv')

# blending baseline 산출물
_save_unit(blend_oof,  y_oof.index,  y_oof,  'oof_unit_blend.csv')
_save_unit(blend_val,  y_val.index,  y_val,  'val_unit_blend.csv')
_save_unit(blend_test, y_test.index, y_test, 'test_unit_blend.csv')

single_df.to_csv(os.path.join(OUT_DIR, 'single_base_rmse.csv'), index=False)
C_oof.to_csv(os.path.join(OUT_DIR, 'residual_corr_oof.csv'))
C_test.to_csv(os.path.join(OUT_DIR, 'residual_corr_test.csv'))
comparison.to_csv(os.path.join(OUT_DIR, 'comparison.csv'), index=False)

# A_OPT 전용 산출물
if USE_OPTUNA_EXTRA and extra_imp_df is not None:
    extra_imp_df.to_csv(os.path.join(OUT_DIR, 'extra_importance.csv'), index=False)
    with open(os.path.join(OUT_DIR, 'best_extra_config.json'), 'w', encoding='utf-8') as f:
        json.dump(OPTUNA_BEST_EXTRA, f, indent=2, ensure_ascii=False, default=str)
    print(f'  [A_OPT] extra_importance.csv ({len(extra_imp_df)} feat) + best_extra_config.json 저장')

# coef는 base_names + extra_feat_names 순서로 매칭
coef_names = list(P_oof.columns)
meta = {
    'exp_tag':         EXP_TAG,
    'options': {
        'USE_EXTRA':       USE_EXTRA,
        'EXTRA_K':         EXTRA_K if USE_EXTRA else None,
        'EXTRA_AGG':       EXTRA_AGG if USE_EXTRA else None,
        'IMPORTANCE_SOURCE': IMPORTANCE_SOURCE if USE_EXTRA else None,
        'extra_feat_names': extra_feat_names if (USE_EXTRA or USE_OPTUNA_EXTRA) else None,
        'USE_OPTUNA_EXTRA':    USE_OPTUNA_EXTRA,
        'OPTUNA_N_TRIALS':     OPTUNA_N_TRIALS if USE_OPTUNA_EXTRA else None,
        'OPTUNA_TIMEOUT':      OPTUNA_TIMEOUT if USE_OPTUNA_EXTRA else None,
        'OPTUNA_BEST_EXTRA':   OPTUNA_BEST_EXTRA,
        'USE_ZERO_CLIP':       USE_ZERO_CLIP,
        'zero_clip_applied':   zero_clip_applied,
        'zero_clip_best_th':   zero_clip_best_th,
        'rmse_pre_zero_clip':  {
            'oof':  rmse_stack_oof_pre,
            'val':  rmse_stack_val_pre,
            'test': rmse_stack_test_pre,
        },
    },
    'pool_models':     list(available.keys()),
    'pool_paths':      {k: available[k] for k in available},
    'missing_models':  list(missing.keys()),
    'meta_learner': {
        'type':       'ElasticNetCV(StandardScaler)',
        'alpha':      best_alpha,
        'l1_ratio':   best_l1_ratio,
        'coef':       {n: float(c) for n, c in zip(coef_names, stack_coef)},
        'intercept':  stack_intercept,
        'positive':   False,
        'max_iter':   20000,
        'cv':         '5-fold KFold(shuffle=True, random_state=SEED)',
        'n_active':   int(n_active),
        'n_negative': int(n_neg),
    },
    'blending_slsqp': {
        'weights':    {n: float(w) for n, w in zip(_base_names, w_blend)},
        'converged':  bool(res.success),
        'rmse_oof':   rmse_blend_oof,
        'rmse_val':   rmse_blend_val,
        'rmse_test':  rmse_blend_test,
    },
    'stacking': {
        'rmse_oof':  rmse_stack_oof,
        'rmse_val':  rmse_stack_val,
        'rmse_test': rmse_stack_test,
        'segment_rmse': segment_rmse,
    },
    'baseline_stacking_11base': {
        'rmse_val':  STACKING_11BASE_VAL,
        'rmse_test': STACKING_11BASE_TEST,
    },
    'delta_vs_11base': {
        'val':  rmse_stack_val  - STACKING_11BASE_VAL,
        'test': rmse_stack_test - STACKING_11BASE_TEST,
    },
    'single_base': single_df.to_dict(orient='records'),
    'min_residual_corr_pair': {
        'a': min_pair[0], 'b': min_pair[1], 'corr_oof': float(min_pair[2]),
    },
    'SEED': int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

OUT_DIR = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\04_stacking\base

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\04_stacking\base
  comparison.csv                         0.4 KB
  meta.json                             11.2 KB
  oof_unit_blend.csv                   972.0 KB
  oof_unit_stack.csv                   972.6 KB
  residual_corr_oof.csv                 11.0 KB
  residual_corr_test.csv                11.0 KB
  single_base_rmse.csv                   1.9 KB
  test_unit_blend.csv                  324.0 KB
  test_unit_stack.csv                  324.2 KB
  val_unit_blend.csv                   324.0 KB
  val_unit_stack.csv                   324.1 KB


## 9. 요약

In [15]:
print('=' * 100)
print(f' Stacking Full Pool ({K}-base) — 결과 요약')
print('=' * 100)
print(comparison.to_string(index=False, float_format='%.6f'))
print('-' * 100)
print(f'  Meta best alpha    : {best_alpha:.4e}')
print(f'  Meta best l1_ratio : {best_l1_ratio:.4f}')
print(f'  Meta intercept     : {stack_intercept:+.6f}')
print(f'  활성 base          : {n_active}/{K} (음수 weight {n_neg}개 = corrector)')
print(f'  최저 residual corr : {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')
if 'nn_ft' in available:
    nn_idx = list(available).index('nn_ft')
    nn_coef = stack_coef[nn_idx]
    nn_min_corr = C_oof['nn_ft'].drop('nn_ft').min()
    print(f'  NN coef            : {nn_coef:+.6f}  ({"활성" if abs(nn_coef) > 1e-9 else "흡수됨"})')
    print(f'  NN 최저 잔차 corr  : {nn_min_corr:.4f}')
print('=' * 100)
# strategy_common §14: stacking_11base 대비 Δ / paradigm 결론은 진단 스크립트 영역.
# meta.json 의 'delta_vs_11base' 필드로 기록 (cell-save) → 정량 비교는 산출물에서.

 Stacking Full Pool (24-base) — 결과 요약
                       method      oof      val     test
 best single (OOF) [zit_only] 0.005497 0.005704 0.008411
 best single (val) [zit_only] 0.005497 0.005704 0.008411
         stacking_11base (기존)      NaN 0.005701 0.008408
     Blending SLSQP (24 base) 0.005496 0.005707 0.008412
Stacking ElasticNet (24 base) 0.005493 0.005704 0.008410
----------------------------------------------------------------------------------------------------
  Meta best alpha    : 1.7433e-05
  Meta best l1_ratio : 0.7000
  Meta intercept     : +0.002481
  활성 base          : 10/24 (음수 weight 3개 = corrector)
  최저 residual corr : bag_zit ↔ reg__enet = 0.9873
